# tensor-unbind — ex5: decompose rays, evaluate at parameter t

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five unbind patterns that ramp from default-dim → explicit-dim → equivalence-with-`select` → tuple-destructure → ray-equation evaluation. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-unbind`**, which bridges to the bank subtopic `Numpy: Indexing and selection` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Unbind — quick refresher

**What it does.** `torch.unbind(x, dim=k)` returns a tuple of `x.shape[k]` tensors, each with axis `k` removed. The result is a *Python tuple* (not a tensor) of *views* (no copy).

**Default dim is 0.** `x.unbind()` peels along axis 0; `x.unbind(dim=1)` peels along axis 1.

**Idiomatic destructure.** `origin, direction = rays.unbind(dim=1)` is the canonical way to split a `(N, 2, 3)` rays tensor into two `(N, 3)` named components.

**Equivalence.** `x.unbind(dim=k)[i]` == `x.select(k, i)`. Prefer `select` for picking ONE slice; prefer `unbind` when you want ALL slices.

### Exercise 5 — decompose rays, evaluate at parameter t

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize axis-explicit unbind + tuple destructure to split a rays tensor and evaluate the parametric ray equation.
> Keywords: ray-tracing, origin-direction, param-evaluate, multi-kc
> ```

**KCs targeted:** `unbind-explicit-dim`, `unbind-tuple-destructure`, `unbind-ray-decomposition`

Implement `ex5_evaluate_rays(rays, t_param)`. The canonical Ray Tracing use of `unbind`:

Input `rays`: shape `(N, 2, 3)`. Each ray is a `(2, 3)` block where row 0 is the origin and row 1 is the direction (in 3-D).

Output: `(N, 3)` — for each ray, return `origin + t_param * direction`.

Implementation: use `rays.unbind(dim=1)` to peel out origin and direction as `(N, 3)` tensors, then return the parametric evaluation.

> ⚠️ **Integrative exercise.** Combines 3 KCs (explicit dim, tuple destructure, ray decomposition). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_evaluate_rays(rays: Tensor, t_param: float) -> Tensor:
    origin, direction = rays.unbind(dim=1)
    return origin + t_param * direction


<details><summary>Solution</summary>

```python
def ex5_evaluate_rays(rays: Tensor, t_param: float) -> Tensor:
    origin, direction = rays.unbind(dim=1)
    return origin + t_param * direction
```

**The parametric ray equation: `r(t) = o + t·d`.** This is the most common Ray Tracing computation — given an array of rays and a hit parameter, find the world-space hit point. `unbind(dim=1)` is the cleanest split: `origin` and `direction` come out as named `(N, 3)` tensors with no shape arithmetic.

**Compare:** `rays[:, 0, :] + t * rays[:, 1, :]` works and is equivalent, but the unbind version reads like math. Prefer it when your variables have physical meaning.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()